# Frequências e palavras em contexto

Seminário de Investigação 2026/27 · Universidade de Coimbra

Correr cada célula com **Shift+Enter**. O notebook lê todos os ficheiros `.pdf` e `.txt` da pasta `textos/`. Para carregar um ficheiro no Binder: abrir a pasta `textos` no painel da esquerda e usar *File → Upload*.

## 0. Escolhas

A pasta, a língua (para retirar as palavras vazias) e a palavra a procurar em contexto. Para experimentar com os textos de exemplo em português: `PASTA = "textos_exemplo"` e `LINGUA = "pt"`.

In [ ]:
PASTA = "textos"          # onde estão os ficheiros
LINGUA = "en"             # "en" para inglês, "pt" para português
PALAVRA = "iceman"        # palavra a mostrar em contexto
SEM_REFERENCIAS = True    # ignorar a lista bibliográfica no fim de um artigo

## 1. Ler os textos

Dos PDF extrai-se o texto página a página. Depois limpam-se as ligaduras tipográficas (ﬁ → fi), as palavras partidas no fim da linha e, se pedido, a bibliografia final.

In [ ]:
import re, unicodedata
from collections import Counter
from pathlib import Path

def ler(ficheiro):
    if ficheiro.suffix.lower() == ".pdf":
        from pypdf import PdfReader
        return "\n".join(p.extract_text() or "" for p in PdfReader(ficheiro).pages)
    return ficheiro.read_text(encoding="utf-8")

def limpar(texto):
    texto = unicodedata.normalize("NFKC", texto)                # ligaduras: ﬁ -> fi
    texto = re.sub(r"-\n\s*(\w)", r"\1", texto)                 # palavras partidas no fim da linha
    texto = re.sub(r"Author.s personal copy", " ", texto)       # cabeçalho repetido de algumas separatas
    if SEM_REFERENCIAS:
        partes = re.split(r"\n\s*(?:References|Referências|Bibliografia)\s*\n", texto)
        if len(partes) > 1:
            texto = "\n".join(partes[:-1])
    return texto

ficheiros = sorted(f for f in Path(PASTA).glob("*") if f.suffix.lower() in (".pdf", ".txt"))
if not ficheiros:
    print(f"A pasta '{PASTA}' não tem ficheiros .pdf ou .txt. Carreguem um ficheiro (File → Upload) e voltem a correr esta célula.")

def palavras(texto):
    return re.findall(r"[a-záàâãéêíóôõúüçäöüß]+(?:-[a-záàâãéêíóôõúüçäöüß]+)*", texto.lower())

tokens = {f.name: palavras(limpar(ler(f))) for f in ficheiros}
for nome, t in tokens.items():
    print(f"{nome:<40}{len(t):>7} palavras")

## 2. Palavras mais frequentes

Retiram-se as palavras vazias (artigos, preposições, conjunções), que seriam sempre as mais frequentes, e algumas abreviaturas académicas (*et al.*, *fig.*). As listas podem ser editadas.

In [ ]:
import matplotlib.pyplot as plt

VAZIAS = {
    "en": set("""a about above after again against all also am an and any are as at be because been before being
    below between both but by can could did do does doing down during each few for from further had has have having
    he her here hers him his how i if in into is it its itself just me more most my no nor not of off on once only or
    other our out over own same she should so some such than that the their them then there these they this those
    through to too under until up very was we were what when where which while who whom why will with would you your
    et al fig figs tab cf ca may within however thus although well since per one two three""".split()),
    "pt": set("""a à ao aos as às até com como contra da das de dela dele deles do dos e é ela elas ele eles em
    entre era eram essa essas esse esses esta estas este estes está estão eu foi foram há isso isto
    já lhe lhes mais mas me mesmo meu minha muito na nas não nem no nos nós num numa o os ou para
    pela pelas pelo pelos por porque quando que quem se sem ser seu seus sua suas só também te tem
    têm ter um uma umas uns vos sobre são sido seja será ainda assim depois onde qual quais tão
    tinha tinham todo toda todos todas outro outra outros outras ali aqui lá então pois cada""".split()),
}[LINGUA]

contagem = Counter(p for t in tokens.values() for p in t if p not in VAZIAS and len(p) > 2)
top = contagem.most_common(20)

plt.figure(figsize=(8, 6))
plt.barh([p for p, _ in reversed(top)], [n for _, n in reversed(top)], color="#595959")
plt.title("As 20 palavras mais frequentes")
plt.xlabel("Ocorrências")
plt.tight_layout()
plt.show()

## 3. Palavras em contexto (KWIC)

Cada ocorrência da `PALAVRA` escolhida na célula 0, com seis palavras antes e depois.

In [ ]:
JANELA = 6
encontradas = 0
for nome, t in tokens.items():
    for i, p in enumerate(t):
        if p == PALAVRA.lower():
            encontradas += 1
            esquerda = " ".join(t[max(0, i - JANELA):i])
            direita = " ".join(t[i + 1:i + 1 + JANELA])
            print(f"{esquerda:>50}  [{p}]  {direita}")
print(f"\n{encontradas} ocorrências de '{PALAVRA}'")